In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [3]:
df=pd.read_csv("train.csv",usecols=["Age","Fare","Survived"])
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [4]:
df.isnull().sum()

Survived      0
Age         177
Fare          0
dtype: int64

In [5]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=["Survived"]),df["Survived"],test_size=0.2,random_state=45)

In [6]:
si=SimpleImputer()
X_train_age=si.fit_transform(X_train[["Age"]])
X_test_age=si.transform(X_test[["Age"]])

In [7]:
X_train_arr=np.hstack([X_train[["Fare"]],X_train_age])
X_test_arr=np.hstack([X_test[["Fare"]],X_test_age])

In [8]:
X_train=pd.DataFrame(X_train_arr,columns=X_train.columns)
X_test=pd.DataFrame(X_test_arr,columns=X_test.columns)
X_test.head()


,Age,Fare
0,52.5542,37.0
1,8.4042,42.0
2,26.0000,29.0
3,56.4958,28.0
4,34.3750,48.0


In [9]:
clf=DecisionTreeClassifier()
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)

In [10]:
accuracy_score(y_test,y_pred)

0.6312849162011173

In [12]:
clf=DecisionTreeClassifier()
cross_val=cross_val_score(clf,X_train,y_train,cv=10)
np.mean(cross_val)

np.float64(0.6305359937402191)

In [13]:
kbin_age=KBinsDiscretizer(n_bins=10,encode='ordinal',strategy='quantile')
kbin_fare=KBinsDiscretizer(n_bins=10,encode='ordinal',strategy='quantile')

In [14]:
trf=ColumnTransformer(transformers=[
    ("first",kbin_age,[0]),
    ("second",kbin_fare,[1])
],remainder="passthrough")

In [15]:
trf.fit(X_train)
X_train_trf=trf.transform(X_train)
X_test_trf=trf.transform(X_test)

C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are r

In [16]:
X_train_trf

array([[9., 6.],
       [1., 3.],
       [7., 0.],
       ...,
       [9., 8.],
       [8., 5.],
       [2., 7.]], shape=(712, 2))

In [17]:
trf.named_transformers_

{'first': KBinsDiscretizer(encode='ordinal', n_bins=10),
 'second': KBinsDiscretizer(encode='ordinal', n_bins=10)}

In [19]:
trf.named_transformers_["first"].n_bins_

array([10])

In [20]:
output=pd.DataFrame({
    "Age":X_train["Age"],
    "Age_trf":X_train_trf[:,0],
    "Fare":X_train["Fare"],
    "Fare_trf":X_train_trf[:,1],
})

In [22]:
output.sample(5)

,Age,Age_trf,Fare,Fare_trf
266,7.8958,2.0,29.93956,5.0
419,13.4167,4.0,4.00000,0.0
562,13.0000,4.0,28.00000,3.0
687,7.7500,1.0,29.93956,5.0
186,7.5208,0.0,22.00000,2.0


In [32]:
def discetise(no_bin,strategy):
    trf=ColumnTransformer(transformers=[
        ("first",KBinsDiscretizer(n_bins=no_bin,encode="ordinal",strategy=strategy),[0]),
        ("second",KBinsDiscretizer(n_bins=no_bin,encode="ordinal",strategy=strategy),[1])
    ],remainder="passthrough")
    trf.fit(X_train)
    X_train_trf=trf.transform(X_train)
    X_test_trf=trf.transform(X_test)
    clf=DecisionTreeClassifier()
    clf.fit(X_train_trf,y_train)
    y_pred=clf.predict(X_test_trf)
    # print(accuracy_score(y_test,y_pred))
    clf=DecisionTreeClassifier()
    cross_val=cross_val_score(clf,X_train_trf,y_train,cv=10)
    print(np.mean(cross_val))    
    

In [34]:
discetise(10,"quantile")

0.6587245696400625


C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\lenovo\anaconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are r